# External Framework Validation — FEST (Bank Marketing)

Runs in the isolated `tese_fest` conda environment (installed from the public
repository). Distance-based privacy (DCR, NNDR, NNAA), similarity and utility
metrics. Mirrors the Dataset 1 FEST notebook, adapted to the Bank feature space.
As in Dataset 1, the correlation metric is expected to return NaN because the
minority reference has a constant target.

## 0. Configuration and paths

In [1]:
from pathlib import Path
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

RANDOM_STATE = 123

# Folder with the CSV files, relative to this notebook (adjust if needed)
DATA_DIR = Path("data")

assert DATA_DIR.exists(), f"Folder not found: {DATA_DIR}"
print("Data folder:", DATA_DIR)

Data folder: data


## 1. Canonical column definitions (Bank Marketing)

In [2]:
TARGET = "y"

FEATURES = ['age','job','marital','education','default','housing','loan','contact',
            'month','day_of_week','campaign','previous','poutcome','emp.var.rate',
            'cons.price.idx','cons.conf.idx','euribor3m','nr.employed','was_contacted']

# Categorical columns: the 11 nominal categoricals (incl. the binary flag).
# Bank Marketing has NO ordinal columns, so there is a single, unambiguous
# categorical treatment (no Variant A/B, unlike the PAY_X columns of Dataset 1).
CAT_COLS = ['job','marital','education','default','housing','loan','contact',
            'month','day_of_week','poutcome','was_contacted']
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

assert len(FEATURES) == 19, f"FEATURES should have 19, got {len(FEATURES)}"
assert TARGET not in FEATURES, "y must not be in FEATURES"
assert len(CAT_COLS) == 11 and len(NUM_COLS) == 8
print("FEATURES:", len(FEATURES), "| CAT:", len(CAT_COLS), "| NUM:", len(NUM_COLS))

ALL_COLS = FEATURES + [TARGET]
print("Working schema:", len(ALL_COLS), "columns")

FEATURES: 19 | CAT: 11 | NUM: 8
Working schema: 20 columns


## 2. Helper: align a dataframe to the canonical schema

In [3]:
def align(df, name):
    d = df.copy()
    if TARGET not in d.columns:
        d[TARGET] = 1   # minority reference: all y == 1
    d = d[ALL_COLS]
    d[NUM_COLS] = d[NUM_COLS].astype("float64")
    d[TARGET] = d[TARGET].astype(int)
    assert list(d.columns) == ALL_COLS
    print(f"{name:<16} -> {d.shape}")
    return d

## 3. Load the real and synthetic datasets

In [4]:
real      = align(pd.read_csv(DATA_DIR / "bank_real_minority_train.csv"),  "real (min)")
syn_ctgan = align(pd.read_csv(DATA_DIR / "bank_synthetic_ctgan_final.csv"), "syn ctgan")
syn_ctab  = align(pd.read_csv(DATA_DIR / "bank_synthetic_ctab_final.csv"),  "syn ctabgan+")

for d in (syn_ctgan, syn_ctab):
    assert list(d.columns) == list(real.columns)
print("\nColumns aligned.")

real (min)       -> (3248, 20)
syn ctgan        -> (22335, 20)
syn ctabgan+     -> (22335, 20)

Columns aligned.


## 4. Privacy metrics (distance-based): DCR, NNDR, NNAA

In [5]:
from privacy_utility_framework.metrics.privacy import PrivacyMetricManager
from privacy_utility_framework.metrics.privacy.distance import (
    DCRCalculator, NNDRCalculator, AdversarialAccuracyCalculator,
)

def privacy_eval(syn, name):
    print(f"\n{'='*55}\n{name}\n{'='*55}")
    p = PrivacyMetricManager()
    p.add_metric([
        DCRCalculator(real, syn, original_name="real_min",
                      synthetic_name=name, preprocess=True),
        NNDRCalculator(real, syn, original_name="real_min",
                       synthetic_name=name, preprocess=True),
        AdversarialAccuracyCalculator(real, syn, original_name="real_min",
                                      synthetic_name=name, preprocess=True,
                                      backend="auto"),
    ])
    results = p.evaluate_all()
    row = {"model": name}
    for key, value in results.items():
        print(f"  {key}: {value}")
        row[str(key)] = value
    return row

priv_rows = [privacy_eval(syn_ctgan, "CTGAN"),
             privacy_eval(syn_ctab, "CTAB-GAN+")]


CTGAN
  DCRCalculator('real_min', 'CTGAN'): 1.7381961565520376
  NNDRCalculator('real_min', 'CTGAN'): 0.8352868370788757
  AdversarialAccuracyCalculator('real_min', 'CTGAN'): 0.7268524599112705

CTAB-GAN+
  DCRCalculator('real_min', 'CTAB-GAN+'): 1.7853706717497864
  NNDRCalculator('real_min', 'CTAB-GAN+'): 0.846425201406634
  AdversarialAccuracyCalculator('real_min', 'CTAB-GAN+'): 0.7318548529390683


## 5. Similarity and utility metrics
The correlation metric is expected to return NaN (constant target in the
minority-only reference), exactly as in Dataset 1.

In [6]:
from privacy_utility_framework.metrics.utility.utility_metric_manager import (
    UtilityMetricManager,
)
from privacy_utility_framework.metrics.utility.statistical import (
    BasicStatsCalculator, CorrelationCalculator, JSCalculator,
    KSCalculator, MICalculator,
)

def utility_eval(syn, name):
    print(f"\n{'='*55}\n{name}\n{'='*55}")
    p = UtilityMetricManager()
    p.add_metric([
        KSCalculator(real, syn, original_name="real_min", synthetic_name=name),
        CorrelationCalculator(real, syn, original_name="real_min", synthetic_name=name),
        MICalculator(real, syn, original_name="real_min", synthetic_name=name),
        JSCalculator(real, syn, original_name="real_min", synthetic_name=name),
        BasicStatsCalculator(real, syn, original_name="real_min", synthetic_name=name),
    ])
    results = p.evaluate_all()
    row = {"model": name}
    for key, value in results.items():
        print(f"  {key}: {value}")
        row[str(key)] = value
    return row

util_rows = [utility_eval(syn_ctgan, "CTGAN"),
             utility_eval(syn_ctab, "CTAB-GAN+")]


CTGAN
Method CorrelationMethod.PEARSON was used.
  KSCalculator('real_min', 'CTGAN'): 0.9515461627139375
  CorrelationCalculator('real_min', 'CTGAN'): nan
  MICalculator('real_min', 'CTGAN'): 0.9913333324405551
  JSCalculator('real_min', 'CTGAN'): 0.9452580078159466
  BasicStatsCalculator('real_min', 'CTGAN'): {'mean': 0.033360965205173204, 'median': 0.002989540849321025, 'var': 0.018517690659254724}

CTAB-GAN+
Method CorrelationMethod.PEARSON was used.
  KSCalculator('real_min', 'CTAB-GAN+'): 0.9405262094528617
  CorrelationCalculator('real_min', 'CTAB-GAN+'): nan
  MICalculator('real_min', 'CTAB-GAN+'): 0.9912399122673422
  JSCalculator('real_min', 'CTAB-GAN+'): 0.9326142506567184
  BasicStatsCalculator('real_min', 'CTAB-GAN+'): {'mean': 0.04019708602378267, 'median': 0.022630395569995326, 'var': 0.022872734583121803}


## 6. Wasserstein distance (sampled)
200 records per sample, averaged over 10 samples, with a fixed seed so that the
value is reproducible. It is a distance: lower = more similar.

In [7]:
from privacy_utility_framework.metrics.utility.statistical import (
    WassersteinCalculator, WassersteinMethod,
)

def wasserstein_eval(syn, name):
    calc = WassersteinCalculator(real, syn)
    np.random.seed(RANDOM_STATE)   # FEST samples with np.random: fix it for reproducibility
    res = calc.evaluate(metric=WassersteinMethod.WASSERSTEIN_SAMPLE,
                        n_samples=200, n_iterations=10)
    print(f"{name}: Wasserstein (sample) = {res:.4f}")
    return {"model": name, "Wasserstein": float(res)}

ws_rows = [wasserstein_eval(syn_ctgan, "CTGAN"),
           wasserstein_eval(syn_ctab, "CTAB-GAN+")]

Sampled Wasserstein Distance: 2.4165947689041447,                     with n_samples=200, n_iterations=10
CTGAN: Wasserstein (sample) = 2.4166
Sampled Wasserstein Distance: 2.442494555965294,                     with n_samples=200, n_iterations=10
CTAB-GAN+: Wasserstein (sample) = 2.4425


## 7. Consolidate and save (clean tidy version)

In [8]:
from functools import reduce

def tidy(rows):
    out = []
    for r in rows:
        clean = {"model": r["model"]}
        for k, v in r.items():
            if k == "model":
                continue
            metric = str(k).split("(")[0].replace("Calculator", "")
            clean[metric] = v
        out.append(clean)
    return out

priv_tidy = pd.DataFrame(tidy(priv_rows)).set_index("model")
util_tidy = pd.DataFrame(tidy(util_rows)).set_index("model")
ws_tidy   = pd.DataFrame(ws_rows).set_index("model")

fest_clean = priv_tidy.join(util_tidy).join(ws_tidy).reset_index()
fest_clean.to_csv(DATA_DIR / "bank_fest_all_results_clean.csv", index=False)
print("Saved: bank_fest_all_results_clean.csv\n")
with pd.option_context("display.max_columns", None, "display.width", 200):
    print(fest_clean.to_string(index=False))

Saved: bank_fest_all_results_clean.csv

    model      DCR     NNDR  AdversarialAccuracy       KS  Correlation       MI       JS                                                                                  BasicStats  Wasserstein
    CTGAN 1.738196 0.835287             0.726852 0.951546          NaN 0.991333 0.945258 {'mean': 0.033360965205173204, 'median': 0.002989540849321025, 'var': 0.018517690659254724}     2.416595
CTAB-GAN+ 1.785371 0.846425             0.731855 0.940526          NaN 0.991240 0.932614  {'mean': 0.04019708602378267, 'median': 0.022630395569995326, 'var': 0.022872734583121803}     2.442495


## 8. Diagnose the correlation NaN (constant target)
Confirms that the correlation NaN stems from the single-class minority
reference, not from a data error.

In [9]:
print("Zero-variance columns in real data:")
print([c for c in real.columns if real[c].nunique() <= 1])
print("Zero-variance columns in CTGAN data:")
print([c for c in syn_ctgan.columns if syn_ctgan[c].nunique() <= 1])
print("\nNaNs in real:", real.isna().sum().sum())
print("NaNs in CTGAN:", syn_ctgan.isna().sum().sum())

Zero-variance columns in real data:
['y']
Zero-variance columns in CTGAN data:
['y']

NaNs in real: 0
NaNs in CTGAN: 0


---